---
title: "Machine Learning: Clustering, Density Estimation, and Anomaly Detection"
lang: en
format:
  html:
    toc: true
    toc-depth: 5
    theme: cosmo
    code-fold: true
jupyter: python
---



<div class="blog-language-switch" role="group" aria-label="Article language">
<span aria-current="page">English</span>
<a href="../zh-CN/Machine-Learning/13-clustering-density-anomaly.html" lang="zh-CN" hreflang="zh-CN">中文</a>
</div>

[Back to Machine Learning guideline](Machine Learning.html)



## **Clustering, Density Estimation, and Anomaly Detection**

Supervised learning asks a model to predict a target that is supplied during training. The methods in this chapter begin with a harder scientific situation: there is no target column that states what the answer should be. They instead describe the distribution of the observations themselves.

Three tasks are related but should not be treated as synonyms:

- **Clustering** constructs a partition, hierarchy, or soft assignment that summarizes which observations belong together.
- **Density estimation** models where observations are likely to occur by estimating a probability density $p(\mathbf{x})$.
- **Anomaly detection** ranks observations by how poorly they conform to a specified notion of normality.

An unsupervised result is never produced by the data alone. It is produced by the combination

$$
\text{result}
=f(\text{representation},\text{metric},\text{model assumptions},\text{hyperparameters},\text{sample}).
$$

Changing the unit of one feature, replacing Euclidean distance with cosine distance, or changing a neighbourhood radius can create a different answer even though the rows are unchanged. The absence of labels therefore does **not** mean the absence of assumptions. It makes those assumptions more important because prediction error against a target cannot automatically expose them.

| Analytical question | Typical output | Central risk |
|---|---|---|
| Are there useful groups? | Hard labels, probabilities, or a hierarchy | The algorithm may impose groups that are not substantively real |
| Where is probability mass concentrated? | A fitted density or log-density score | High-dimensional estimates can be unstable and poorly calibrated |
| Which observations deserve investigation? | Anomaly scores and an alert threshold | Rare does not necessarily mean erroneous, harmful, or interesting |

This chapter develops centroid, connectivity, density, graph, and probabilistic views of structure. The aim is not to identify one universally best method. It is to learn how each method defines similarity, what structure it can express, how its output should be diagnosed, and when “no convincing structure” is the most defensible conclusion.



### **What Does a Cluster Mean?**

A **cluster** is a set of observations regarded as more strongly related to one another than to observations outside the set. That definition is deliberately incomplete: “related” can mean close to a centroid, linked by a chain of neighbours, located in the same high-density region, generated by the same probability component, or connected strongly in a graph.

Consider customers described by annual spending and number of purchases. One analysis may seek compact customer segments around representative profiles. Another may seek a thin path from occasional to frequent purchasing. A third may identify dense communities while leaving unusual customers unassigned. These are different mathematical questions, so they can reasonably produce different clusters.

<div class="diagram-scroll">

![Compactness, connectivity, density, and distributional definitions of a cluster.](assets/cluster-definitions.svg){fig-alt="Four panels show clusters defined by distance to a centroid, chains of nearby points, dense regions separated by sparse regions, and overlapping probability components."}

</div>

#### **Similarity, Compactness, Connectivity, and Density**

Before selecting an algorithm, define the **observational unit**, the **representation**, and the **dissimilarity measure**. For numeric vectors, Euclidean distance is

$$
d_2(\mathbf{x}_i,\mathbf{x}_j)
=\sqrt{\sum_{r=1}^{d}(x_{ir}-x_{jr})^2}.
$$

Every feature contributes in its numerical units. A difference of 10,000 dollars can dominate a difference of 20 purchases even if purchasing frequency is more meaningful. Standardization changes feature $r$ to $(x_{ir}-\bar{x}_r)/s_r$, making a one-standard-deviation difference comparable across features. This is a modelling decision, not a universally correct preprocessing step: standardization can also magnify noisy low-variance features.

Cosine similarity instead compares directions,

$$
\operatorname{cos}(\mathbf{x}_i,\mathbf{x}_j)
=\frac{\mathbf{x}_i^\top\mathbf{x}_j}
{\lVert\mathbf{x}_i\rVert_2\lVert\mathbf{x}_j\rVert_2},
$$

and is often useful for sparse text vectors when relative composition matters more than document length. Manhattan distance can be more robust to one large coordinate difference; Hamming or Jaccard dissimilarity can suit binary attributes; mixed numeric and categorical records may require a mixed-type measure such as Gower distance. A cluster statement is interpretable only when the metric has an interpretable meaning.

The main cluster concepts are:

| Cluster concept | Operational meaning | Representative methods | Characteristic limitation |
|---|---|---|---|
| Compactness | Small within-cluster distance to a representative | K-Means, K-Medoids | Prefers roughly convex groups |
| Connectivity | Members can be reached through nearby members | Hierarchical single linkage, spectral methods | Can join groups through a thin bridge |
| Density | Dense regions are separated by sparse regions | DBSCAN, HDBSCAN | Depends on neighbourhood scale |
| Distribution | Members have high probability under one latent component | Gaussian mixture models | Depends on the component family |

Outputs also differ. A **hard partition** assigns each observation to exactly one cluster. A **soft assignment** reports membership probabilities or responsibilities. A hierarchy gives nested groups at many resolutions. Density methods can label some points as noise. None of these output types is automatically more truthful; each represents a different claim.

<details>
<summary><strong>Python: observe how scaling changes nearest neighbours</strong></summary>

```python
import numpy as np
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import pairwise_distances

# Columns are annual spending in dollars and visits per year.
customers = np.array([
    [20_000, 4],
    [21_500, 16],
    [42_000, 5],
    [43_000, 18],
], dtype=float)
names = np.array(["A", "B", "C", "D"])

raw_distances = pairwise_distances(customers)
scaled_customers = StandardScaler().fit_transform(customers)
scaled_distances = pairwise_distances(scaled_customers)

def nearest_other(distance_matrix, index):
    row = distance_matrix[index].copy()
    row[index] = np.inf
    return names[np.argmin(row)]

for index, name in enumerate(names):
    print(
        name,
        "nearest before scaling:", nearest_other(raw_distances, index),
        "| after scaling:", nearest_other(scaled_distances, index),
    )
```

</details>

In the raw representation, spending dominates the geometry. After standardization, visit frequency can change which customer is regarded as nearest. The example does not prove that scaling is correct; it proves that the preprocessing choice is part of the cluster definition.



### **K-Means Clustering**

K-Means represents each cluster by its arithmetic mean, or **centroid**. It is suitable when the intended groups are compact in Euclidean space and a centroid is a meaningful summary. Its popularity comes from a simple objective, efficient iterations, and scalability to large numeric datasets. Those advantages should not be mistaken for a guarantee that every dataset contains K-Means-shaped clusters.

#### **Objective and Lloyd's Algorithm**

Let $z_i\in\{1,\ldots,K\}$ be the cluster assigned to observation $\mathbf{x}_i$, and let $\boldsymbol\mu_k$ be centroid $k$. K-Means minimizes the **within-cluster sum of squared distances**:

$$
J(\mathbf z,\boldsymbol\mu)
=\sum_{i=1}^{n}\left\lVert
\mathbf{x}_i-\boldsymbol\mu_{z_i}
\right\rVert_2^2.
$$

The expression has a concrete interpretation. For each observation, select the centroid indexed by $z_i$, measure the squared Euclidean residual, and add all residuals. Squaring emphasizes distant observations and makes the arithmetic mean the optimal representative when assignments are fixed.

Lloyd's algorithm alternates two exact conditional improvements:

```text
Choose K initial centroids.
repeat
    Assignment step:
        assign every observation to its nearest centroid
    Update step:
        replace every centroid by the mean of its assigned observations
until assignments stop changing or the objective improvement is negligible
```

The assignment step minimizes $J$ while the centroids are fixed. The update step minimizes $J$ while the assignments are fixed. Consequently, the objective cannot increase. However, the joint problem is non-convex: monotonic improvement leads to a **local** optimum, not necessarily the globally best partition.

<div class="diagram-scroll">

![The assignment and centroid-update stages of Lloyd's K-Means algorithm.](assets/kmeans-lloyd-loop.svg){fig-alt="A flow diagram shows initialization followed by repeated assignment of points to the nearest centroid and movement of each centroid to its assigned mean until convergence."}

</div>

<details>
<summary><strong>Python: implement Lloyd's algorithm from first principles</strong></summary>

```python
import numpy as np
from sklearn.datasets import make_blobs

X, _ = make_blobs(
    n_samples=240,
    centers=[(-4, -2), (0, 4), (4, -1)],
    cluster_std=[0.8, 1.0, 0.9],
    random_state=7,
)

def lloyd_kmeans(X, n_clusters, seed=0, max_iter=100):
    rng = np.random.default_rng(seed)
    # Start from distinct observations so each centroid is in data space.
    centroids = X[rng.choice(len(X), size=n_clusters, replace=False)].copy()
    inertia_history = []

    for _ in range(max_iter):
        # Assignment: squared distances from every row to every centroid.
        squared_distances = ((X[:, None, :] - centroids[None, :, :]) ** 2).sum(axis=2)
        labels = squared_distances.argmin(axis=1)
        inertia = squared_distances[np.arange(len(X)), labels].sum()
        inertia_history.append(float(inertia))

        # Update: each non-empty cluster is represented by its arithmetic mean.
        new_centroids = centroids.copy()
        for cluster in range(n_clusters):
            members = X[labels == cluster]
            if len(members) > 0:
                new_centroids[cluster] = members.mean(axis=0)

        if np.allclose(new_centroids, centroids):
            centroids = new_centroids
            break
        centroids = new_centroids

    return labels, centroids, np.array(inertia_history)

labels, centroids, history = lloyd_kmeans(X, n_clusters=3, seed=11)
print("iterations:", len(history))
print("objective never increased:", bool(np.all(np.diff(history) <= 1e-9)))
print("final inertia:", round(history[-1], 2))
print("centroids:\n", np.round(centroids, 2))
```

</details>

The reported **inertia** is exactly $J$. It is useful for comparing runs with the same representation and $K$, but it always weakly decreases as $K$ grows. It therefore cannot, by itself, prove the correct number of clusters.

#### **Initialization and K-Means++**

Poor starting centroids can send Lloyd's algorithm to a poor local optimum. Running many random starts and retaining the lowest-inertia result reduces this risk. **K-Means++** creates more informative starts by spreading centroids through the data:

1. choose the first centroid uniformly from the observations;
2. compute $D(\mathbf{x})$, the distance from each observation to its nearest chosen centroid;
3. choose the next centroid with probability proportional to $D(\mathbf{x})^2$;
4. repeat until $K$ centroids have been selected, then run Lloyd's algorithm.

Points far from existing centroids are more likely to seed a new region, but the procedure remains stochastic. In practice, use K-Means++ together with multiple initializations and record the random seed.

<details>
<summary><strong>Python: compare random and K-Means++ starts</strong></summary>

```python
import numpy as np
from sklearn.cluster import KMeans
from sklearn.datasets import make_blobs

X, _ = make_blobs(
    n_samples=700,
    centers=6,
    cluster_std=[0.7, 1.8, 0.6, 1.2, 0.8, 1.5],
    random_state=12,
)

def inertias_for(initialization):
    scores = []
    for seed in range(20):
        # n_init=1 exposes the variability of one initialization.
        model = KMeans(
            n_clusters=6,
            init=initialization,
            n_init=1,
            random_state=seed,
        )
        model.fit(X)
        scores.append(model.inertia_)
    return np.array(scores)

random_scores = inertias_for("random")
plus_scores = inertias_for("k-means++")

for name, scores in [("random", random_scores), ("k-means++", plus_scores)]:
    print(
        f"{name:11s}",
        "best=", round(scores.min(), 1),
        "median=", round(np.median(scores), 1),
        "worst=", round(scores.max(), 1),
    )
```

</details>

#### **Failure Modes**

K-Means implicitly favors clusters that are roughly convex, similarly dispersed, and adequately represented by their means. It can fail for curved manifolds, unequal density, strongly unequal sizes, non-Euclidean similarity, categorical data, and datasets containing influential outliers.

<div class="diagram-scroll">

![Situations in which K-Means can impose misleading partitions.](assets/kmeans-failure-modes.svg){fig-alt="Four panels illustrate successful compact clusters and failure on crescent shapes, unequal density, and outliers that pull centroids."}

</div>

Feature scaling is especially consequential because the objective uses squared distance. Outliers have squared influence and can drag a centroid away from the dense core. K-Medoids can replace means with observed representatives for greater robustness, although at greater computational cost. For very large samples with appropriate geometry, Mini-Batch K-Means trades exact updates for faster stochastic ones.

<details>
<summary><strong>Python: compare K-Means on compact blobs and curved moons</strong></summary>

```python
from sklearn.cluster import KMeans
from sklearn.datasets import make_blobs, make_moons
from sklearn.metrics import adjusted_rand_score, silhouette_score
from sklearn.preprocessing import StandardScaler

datasets = {
    "compact blobs": make_blobs(n_samples=500, centers=2, cluster_std=0.8, random_state=4),
    "curved moons": make_moons(n_samples=500, noise=0.07, random_state=4),
}

for name, (X, synthetic_truth) in datasets.items():
    X = StandardScaler().fit_transform(X)
    labels = KMeans(n_clusters=2, n_init=20, random_state=4).fit_predict(X)
    print(
        name,
        "| silhouette:", round(silhouette_score(X, labels), 3),
        "| synthetic ARI:", round(adjusted_rand_score(synthetic_truth, labels), 3),
    )
```

</details>

The synthetic labels are used here only because the data generator exposes the intended geometry. In a real unsupervised task, those labels usually do not exist. The experiment shows why a respectable compactness score cannot establish that a partition matches the structure relevant to the application.



### **Hierarchical Clustering**

Hierarchical clustering represents grouping at multiple resolutions rather than committing immediately to one flat partition. The result is a tree of nested sets: nearby observations merge low in the tree, while broad groups meet only at greater dissimilarity. This is useful when a domain naturally contains subtypes inside larger categories, or when the appropriate number of clusters is not known in advance.

#### **Agglomerative and Divisive Methods**

**Agglomerative** clustering starts with every observation in its own cluster and repeatedly merges the two closest clusters. **Divisive** clustering starts with all observations together and recursively splits them. Agglomerative methods are much more common because the merge operation is simple and mature implementations are widely available.

An agglomerative procedure is:

```text
Start with n singleton clusters.
Compute dissimilarities between all current clusters.
repeat until only one cluster remains
    merge the pair with the smallest linkage dissimilarity
    update dissimilarities between the new cluster and all others
    record the merge dissimilarity
```

The word **linkage** is crucial. Point-to-point distance does not by itself define distance between two sets. Different linkage rules encode different meanings of a coherent group and can produce very different hierarchies from the same distance matrix.

#### **Linkage Criteria and Dendrograms**

For clusters $A$ and $B$ with pointwise distance $d(\mathbf{x}_i,\mathbf{x}_j)$:

$$
d_{\text{single}}(A,B)=\min_{i\in A,j\in B}d(\mathbf{x}_i,\mathbf{x}_j),
$$

$$
d_{\text{complete}}(A,B)=\max_{i\in A,j\in B}d(\mathbf{x}_i,\mathbf{x}_j),
$$

$$
d_{\text{average}}(A,B)
=\frac{1}{|A||B|}\sum_{i\in A}\sum_{j\in B}d(\mathbf{x}_i,\mathbf{x}_j).
$$

Single linkage asks whether **any** pair connects the clusters and can recover elongated shapes, but it can chain groups together through a thin bridge. Complete linkage controls the worst pairwise separation and tends to form compact groups, but it is sensitive to extreme points. Average linkage is a compromise. Ward linkage is specialized to squared Euclidean geometry and merges the pair causing the smallest increase in within-cluster sum of squares:

$$
\Delta(A,B)
=\frac{|A||B|}{|A|+|B|}
\left\lVert\boldsymbol\mu_A-\boldsymbol\mu_B\right\rVert_2^2.
$$

The size factor prevents a merge decision from depending only on centroid distance. Ward's method is often K-Means-like in its preference for compact groups, whereas single linkage expresses connectivity.

<div class="diagram-scroll">

![Linkage criteria and the interpretation of a dendrogram cut.](assets/linkage-dendrogram.svg){fig-alt="The diagram compares single, complete, average, and Ward linkage, then shows a dendrogram whose horizontal cut creates a flat partition."}

</div>

A **dendrogram** plots the sequence of merges. Leaves are observations or preliminary groups, and the vertical position of a merge records its linkage dissimilarity. Cutting the tree horizontally gives a flat partition. The left-to-right order of leaves is not a one-dimensional measurement, and the merge height is not a probability or confidence level. A large vertical gap can suggest a stable resolution, but it does not prove that the corresponding groups are substantively real.

Straightforward hierarchical clustering stores an $n\times n$ dissimilarity structure, so memory can become $O(n^2)$. This makes it attractive for exploratory analysis of moderate samples, but unsuitable for millions of observations without approximation, connectivity constraints, or prior compression.

<details>
<summary><strong>Python: inspect merge history under different linkages</strong></summary>

```python
import numpy as np
from scipy.cluster.hierarchy import linkage, fcluster
from sklearn.datasets import make_blobs
from sklearn.metrics import adjusted_rand_score

X, synthetic_truth = make_blobs(
    n_samples=60,
    centers=[(-3, 0), (0, 0), (3, 0)],
    cluster_std=[0.35, 0.85, 0.35],
    random_state=9,
)

for method in ["single", "complete", "average", "ward"]:
    # Each row of Z is: left child, right child, merge height, merged size.
    Z = linkage(X, method=method, metric="euclidean")
    labels = fcluster(Z, t=3, criterion="maxclust") - 1
    last_merges = np.round(Z[-3:, 2], 3)
    print(
        f"{method:8s}",
        "last merge heights:", last_merges.tolist(),
        "| synthetic ARI:", round(adjusted_rand_score(synthetic_truth, labels), 3),
    )
```

</details>

The external synthetic labels make the linkage differences visible in this controlled example. In an unlabeled application, compare dendrogram structure, sensitivity to scaling and linkage, cluster stability, and domain interpretation rather than selecting the most visually pleasing tree.



### **Density-Based Clustering**

Density-based methods define a cluster as a region containing many nearby observations, separated from other such regions by sparse space. Unlike K-Means, they can recover non-convex groups and can leave observations unassigned as noise. The price is that density must be defined relative to a neighbourhood scale and a minimum amount of local support.

#### **DBSCAN**

DBSCAN uses two principal hyperparameters:

- $\varepsilon$ (`eps`) is the neighbourhood radius;
- `min_samples` is the minimum number of observations, including the point itself, required for a dense neighbourhood.

The $\varepsilon$-neighbourhood of point $\mathbf{x}_i$ is

$$
N_{\varepsilon}(\mathbf{x}_i)
=\{\mathbf{x}_j:d(\mathbf{x}_i,\mathbf{x}_j)\leq\varepsilon\}.
$$

A **core point** satisfies $|N_{\varepsilon}(\mathbf{x}_i)|\geq m$, where $m$ is `min_samples`. A **border point** has too few neighbours to be core but lies in a core point's neighbourhood. A **noise point** is neither core nor density-reachable from one. Clusters are maximal sets connected through chains of core neighbourhoods, with reachable border points attached.

```text
Mark every observation unvisited.
For each unvisited observation x:
    find its epsilon-neighbourhood
    if x is not a core point, provisionally mark it as noise
    otherwise start a new cluster and expand it through core neighbours
        add every density-reachable core point
        attach border points encountered during expansion
```

<div class="diagram-scroll">

![Core, border, and noise points in DBSCAN.](assets/dbscan-core-border-noise.svg){fig-alt="A neighbourhood circle illustrates a core point with sufficient neighbours, a border point reachable from the dense region, and an isolated noise point."}

</div>

A smaller `eps` fragments clusters and marks more points as noise. A larger `eps` joins regions and may bridge distinct clusters. Increasing `min_samples` demands stronger density support. A sorted distance to each point's $k$-th nearest neighbour can reveal candidate scales, but an apparent elbow is only a diagnostic and may not exist when densities vary.

With a spatial index, neighbourhood queries can be efficient in low dimensions. In high dimensions, distances concentrate and tree indexes deteriorate; DBSCAN can approach quadratic work or memory. Scaling, metric choice, duplicates, and sample density all matter.

<details>
<summary><strong>Python: recover curved clusters and identify noise with DBSCAN</strong></summary>

```python
import numpy as np
from sklearn.cluster import DBSCAN
from sklearn.datasets import make_moons
from sklearn.metrics import adjusted_rand_score
from sklearn.preprocessing import StandardScaler

X, synthetic_truth = make_moons(n_samples=500, noise=0.065, random_state=13)
rng = np.random.default_rng(13)
background_noise = rng.uniform(low=[-1.5, -1.0], high=[2.5, 1.5], size=(35, 2))
X = np.vstack([X, background_noise])
X = StandardScaler().fit_transform(X)

labels = DBSCAN(eps=0.18, min_samples=8).fit_predict(X)
core_mask = labels != -1
cluster_count = len(set(labels)) - (1 if -1 in labels else 0)

print("clusters:", cluster_count)
print("noise observations:", int((labels == -1).sum()))
# ARI is calculated only on the generated moon observations.
print(
    "synthetic moon ARI:",
    round(adjusted_rand_score(synthetic_truth, labels[: len(synthetic_truth)]), 3),
)
print("assigned fraction:", round(core_mask.mean(), 3))
```

</details>

The label `-1` means noise, not automatically bad data. A rare but valid event, a transition between regimes, and a measurement error can all receive that label. Interpretation requires returning to the original records.

#### **HDBSCAN and Variable Density**

One global $\varepsilon$ cannot simultaneously preserve a sparse cluster and avoid merging a dense cluster with its surroundings. HDBSCAN addresses this by building a hierarchy over density scales rather than selecting one radius at the beginning.

For a chosen `min_samples`, the **core distance** of point $\mathbf{x}$ is the distance to its `min_samples`-th nearest neighbour. HDBSCAN defines mutual-reachability distance

$$
d_{\text{mr}}(\mathbf{x}_i,\mathbf{x}_j)
=\max\left\{
d_{\text{core}}(\mathbf{x}_i),
d_{\text{core}}(\mathbf{x}_j),
d(\mathbf{x}_i,\mathbf{x}_j)
\right\}.
$$

This stretches distances around isolated points while preserving distances inside dense regions. HDBSCAN then:

1. builds a minimum spanning tree under mutual-reachability distance;
2. converts edge removals across density levels into a cluster hierarchy;
3. condenses branches smaller than `min_cluster_size`;
4. selects clusters that persist over a large density interval.

<div class="diagram-scroll">

![How HDBSCAN converts variable-density data into a stability hierarchy.](assets/hdbscan-stability-tree.svg){fig-alt="A flow diagram shows core distances, mutual-reachability edges, a minimum spanning tree, a condensed hierarchy, and stable cluster selection."}

</div>

`min_cluster_size` expresses the smallest group worth retaining. `min_samples` controls how conservative the density estimate is; larger values generally label more points as noise. HDBSCAN reduces dependence on one radius, but it does not eliminate modelling choices or guarantee recovery of every variable-density structure.

<details>
<summary><strong>Python: compare one-scale DBSCAN with hierarchical density clustering</strong></summary>

```python
import numpy as np
from sklearn.cluster import DBSCAN, HDBSCAN
from sklearn.datasets import make_blobs
from sklearn.metrics import adjusted_rand_score
from sklearn.preprocessing import StandardScaler

# The left group is dense; the right group is much more diffuse.
X, synthetic_truth = make_blobs(
    n_samples=[260, 340],
    centers=[(-3.0, 0.0), (3.0, 0.0)],
    cluster_std=[0.22, 1.15],
    random_state=21,
)
X = StandardScaler().fit_transform(X)

for eps in [0.10, 0.18, 0.30]:
    labels = DBSCAN(eps=eps, min_samples=10).fit_predict(X)
    clusters = len(set(labels)) - (1 if -1 in labels else 0)
    print(
        f"DBSCAN eps={eps:.2f}",
        "clusters=", clusters,
        "noise=", int((labels == -1).sum()),
        "ARI=", round(adjusted_rand_score(synthetic_truth, labels), 3),
    )

hdbscan_labels = HDBSCAN(min_cluster_size=35, min_samples=10).fit_predict(X)
print(
    "HDBSCAN       ",
    "clusters=", len(set(hdbscan_labels)) - (1 if -1 in hdbscan_labels else 0),
    "noise=", int((hdbscan_labels == -1).sum()),
    "ARI=", round(adjusted_rand_score(synthetic_truth, hdbscan_labels), 3),
)
```

</details>

DBSCAN remains an excellent interpretable baseline when one density scale is meaningful. HDBSCAN is preferable when meaningful groups persist at different density levels and a hierarchy plus noise assignments is useful.



### **Spectral Clustering**

Spectral clustering converts observations into a weighted graph. Each observation is a node, and an edge weight $w_{ij}$ expresses similarity. Clustering then seeks weakly connected graph regions rather than compact regions around Euclidean centroids. This makes non-convex structures accessible after an appropriate graph embedding.

Let $\mathbf W$ be the symmetric affinity matrix and let the degree matrix be diagonal with

$$
D_{ii}=\sum_j W_{ij}.
$$

The unnormalized graph Laplacian is $\mathbf L=\mathbf D-\mathbf W$. A common normalized form is

$$
\mathbf L_{\text{sym}}
=\mathbf I-\mathbf D^{-1/2}\mathbf W\mathbf D^{-1/2}.
$$

The smallest-eigenvalue eigenvectors vary slowly across strong graph edges. Using several of them as coordinates maps strongly connected nodes close together. K-Means in that **spectral embedding** can then implement an approximate graph cut that was nonlinear in the original coordinates.

<div class="diagram-scroll">

![The graph construction, Laplacian embedding, and final partition stages of spectral clustering.](assets/spectral-graph-embedding.svg){fig-alt="Four stages show non-convex observations, a local affinity graph, low-frequency Laplacian eigenvectors, and a simple partition in spectral coordinates."}

</div>

The algorithmic pipeline is:

```text
Construct an affinity graph W from observations.
Compute a graph Laplacian L from W and its degree matrix D.
Extract eigenvectors associated with the smallest relevant eigenvalues.
Treat each row of those eigenvectors as an embedded observation.
Cluster the embedded rows, commonly with K-Means.
```

The affinity graph is the main modelling decision. An RBF affinity

$$
w_{ij}=\exp(-\gamma\lVert\mathbf{x}_i-\mathbf{x}_j\rVert_2^2)
$$

connects all pairs with rapidly decaying weights. A $k$-nearest-neighbour graph is sparse and emphasizes local structure. If the graph is disconnected, too dense, or dominated by noisy features, the eigenvectors faithfully encode the wrong graph.

Storing dense affinities costs $O(n^2)$ memory, and eigendecomposition can dominate computation. Sparse neighbour graphs and approximate eigensolvers help, but spectral clustering is generally less scalable than K-Means. It also requires the number of clusters unless an eigengap or downstream criterion provides a defensible choice.

<details>
<summary><strong>Python: compare centroid and graph-based clustering on two moons</strong></summary>

```python
from sklearn.cluster import KMeans, SpectralClustering
from sklearn.datasets import make_moons
from sklearn.metrics import adjusted_rand_score
from sklearn.preprocessing import StandardScaler

X, synthetic_truth = make_moons(n_samples=600, noise=0.07, random_state=8)
X = StandardScaler().fit_transform(X)

kmeans_labels = KMeans(n_clusters=2, n_init=20, random_state=8).fit_predict(X)
spectral_labels = SpectralClustering(
    n_clusters=2,
    affinity="nearest_neighbors",
    n_neighbors=12,
    assign_labels="kmeans",
    n_init=20,
    random_state=8,
).fit_predict(X)

print("K-Means synthetic ARI:", round(adjusted_rand_score(synthetic_truth, kmeans_labels), 3))
print("Spectral synthetic ARI:", round(adjusted_rand_score(synthetic_truth, spectral_labels), 3))
```

</details>

Spectral clustering succeeds here because local connectivity matches the generator's curved geometry. This is not universal superiority: on large, compact blobs, K-Means is simpler, faster, and usually the better engineering choice.



### **Mixture Models and EM**

A mixture model assumes that each observation was generated by one of several latent probability components. Unlike K-Means, it describes an explicit density and can report uncertainty about component membership. A component is a mathematical part of the density; calling it a real-world cluster requires additional interpretation.

#### **Gaussian Mixture Models**

A $K$-component Gaussian mixture model (GMM) has density

$$
p(\mathbf{x})
=\sum_{k=1}^{K}\pi_k
\,\mathcal N(\mathbf{x}\mid\boldsymbol\mu_k,\boldsymbol\Sigma_k),
$$

where $\pi_k\geq0$, $\sum_k\pi_k=1$, and each component has mean $\boldsymbol\mu_k$ and covariance $\boldsymbol\Sigma_k$. The mixing weight $\pi_k$ is the prior probability of selecting component $k$. The covariance controls orientation, scale, and correlation, so full-covariance GMMs can express elliptical groups that K-Means cannot.

Covariance constraints trade flexibility for statistical and computational stability:

| Covariance type | Assumption | Consequence |
|---|---|---|
| Spherical | One variance per component | Round components, few parameters |
| Diagonal | Feature independence within each component | Axis-aligned ellipses |
| Tied | All components share one full covariance | Different means, common shape |
| Full | One unrestricted covariance per component | Flexible but data-hungry |

K-Means can be viewed as a limiting hard-assignment case of equally weighted spherical Gaussian components with a common variance approaching zero. A GMM is more expressive, but additional flexibility increases sensitivity to initialization and overfitting.

#### **Expectation-Maximization**

If the latent component labels $z_i$ were known, parameter estimation would be straightforward. If the parameters were known, component probabilities would be straightforward. **Expectation-Maximization (EM)** alternates these two easier problems.

In the E-step, compute the responsibility of component $k$ for observation $i$:

$$
\gamma_{ik}
=P(z_i=k\mid\mathbf{x}_i)
=\frac{\pi_k\mathcal N(\mathbf{x}_i\mid\boldsymbol\mu_k,\boldsymbol\Sigma_k)}
{\sum_{j=1}^{K}\pi_j\mathcal N(\mathbf{x}_i\mid\boldsymbol\mu_j,\boldsymbol\Sigma_j)}.
$$

The denominator normalizes the component scores, so $\sum_k\gamma_{ik}=1$. In the M-step, let $N_k=\sum_i\gamma_{ik}$ and update

$$
\pi_k=\frac{N_k}{n},
\qquad
\boldsymbol\mu_k
=\frac{1}{N_k}\sum_i\gamma_{ik}\mathbf{x}_i,
$$

$$
\boldsymbol\Sigma_k
=\frac{1}{N_k}\sum_i\gamma_{ik}
(\mathbf{x}_i-\boldsymbol\mu_k)(\mathbf{x}_i-\boldsymbol\mu_k)^\top.
$$

These are weighted estimates: an observation contributes to every component in proportion to its responsibility. EM does not decrease the observed-data log-likelihood, but it can converge to a local optimum. Multiple starts, covariance regularization, and held-out evaluation are therefore important.

<div class="diagram-scroll">

![Soft membership and the E-step/M-step cycle in a Gaussian mixture model.](assets/gmm-em-soft-membership.svg){fig-alt="Overlapping Gaussian components give an ambiguous point two responsibilities, while a loop alternates responsibility calculation and weighted parameter updates."}

</div>

<details>
<summary><strong>Python: implement EM for a one-dimensional Gaussian mixture</strong></summary>

```python
import numpy as np
from scipy.special import logsumexp

rng = np.random.default_rng(5)
X = np.concatenate([
    rng.normal(-2.0, 0.65, size=220),
    rng.normal(2.5, 1.10, size=280),
])

means = np.array([-1.0, 1.0])
variances = np.array([1.0, 1.0])
weights = np.array([0.5, 0.5])
log_likelihoods = []

for _ in range(100):
    # E-step in log space prevents numerical underflow.
    log_components = []
    for weight, mean, variance in zip(weights, means, variances):
        log_density = -0.5 * (
            np.log(2 * np.pi * variance) + (X - mean) ** 2 / variance
        )
        log_components.append(np.log(weight) + log_density)
    log_components = np.column_stack(log_components)
    log_normalizer = logsumexp(log_components, axis=1, keepdims=True)
    responsibilities = np.exp(log_components - log_normalizer)
    log_likelihoods.append(float(log_normalizer.sum()))

    # M-step: update parameters with responsibility-weighted statistics.
    effective_counts = responsibilities.sum(axis=0)
    weights = effective_counts / len(X)
    means = (responsibilities * X[:, None]).sum(axis=0) / effective_counts
    variances = (
        responsibilities * (X[:, None] - means) ** 2
    ).sum(axis=0) / effective_counts
    variances = np.maximum(variances, 1e-6)

print("weights:", np.round(weights, 3))
print("means:", np.round(means, 3))
print("standard deviations:", np.round(np.sqrt(variances), 3))
print("likelihood never decreased:", bool(np.all(np.diff(log_likelihoods) >= -1e-8)))
```

</details>

#### **Soft Cluster Membership**

Responsibilities quantify ambiguity under the fitted model. A point near one component centre may have $(0.99,0.01)$ membership, while a point in an overlap may have $(0.52,0.48)$. This supports risk-aware decisions and reveals uncertain boundaries that hard labels hide.

The number of components can be compared using held-out log-likelihood or information criteria. The Bayesian information criterion is

$$
\operatorname{BIC}
=-2\log p(\mathbf X\mid\widehat\theta)+q\log n,
$$

where $q$ is the number of free parameters. Lower BIC balances fit against complexity. It selects a useful density model under assumptions; it does not discover an ontologically true number of social, biological, or behavioral groups.

<details>
<summary><strong>Python: inspect soft membership and select component count with BIC</strong></summary>

```python
import numpy as np
from sklearn.datasets import make_blobs
from sklearn.mixture import GaussianMixture

X, _ = make_blobs(
    n_samples=700,
    centers=[(-3, 0), (0, 0.5), (3, 0)],
    cluster_std=[0.7, 1.25, 0.8],
    random_state=16,
)

candidates = []
for components in range(1, 7):
    model = GaussianMixture(
        n_components=components,
        covariance_type="full",
        n_init=10,
        reg_covar=1e-6,
        random_state=16,
    ).fit(X)
    candidates.append((model.bic(X), model))

best_bic, best_model = min(candidates, key=lambda item: item[0])
probabilities = best_model.predict_proba(X)
uncertainty = 1 - probabilities.max(axis=1)
most_ambiguous = np.argsort(uncertainty)[-5:][::-1]

print("selected components:", best_model.n_components)
print("BIC:", round(best_bic, 1))
print("five largest uncertainties:", np.round(uncertainty[most_ambiguous], 3))
print("their memberships:\n", np.round(probabilities[most_ambiguous], 3))
```

</details>

Mixture component numbers are not stable semantic names: component labels can permute without changing the density, and one irregular real-world group may require several Gaussian components. Interpret probabilities as model-based responsibilities rather than ground-truth class probabilities.



### **Density Estimation**

Density estimation seeks a function $\widehat p(\mathbf{x})$ that describes how probability mass is distributed through the feature space. It supports likelihood-based comparison, simulation, missing-data models, anomaly scores, and probabilistic components. It is a different goal from clustering: a smooth unimodal density may have no meaningful partition, while one substantive group may require a multimodal density.

A density value is not the probability of observing exactly $\mathbf{x}$. For continuous data, that probability is zero. Probability is obtained by integrating density over a region $A$:

$$
P(\mathbf X\in A)=\int_A p(\mathbf{x})\,d\mathbf{x}.
$$

Density also depends on units. Changing metres to centimetres changes numerical density values because the volume element changes, although probabilities for corresponding physical regions remain the same. Compare log density only under a consistent representation and measure.

#### **Parametric Density Models**

A parametric model assumes that the density belongs to a family indexed by a fixed-dimensional parameter $\theta$. A multivariate Gaussian, for example, uses a mean vector and covariance matrix:

$$
p(\mathbf{x})
=\frac{1}{(2\pi)^{d/2}|\boldsymbol\Sigma|^{1/2}}
\exp\left[-\frac{1}{2}
(\mathbf{x}-\boldsymbol\mu)^\top
\boldsymbol\Sigma^{-1}
(\mathbf{x}-\boldsymbol\mu)
\right].
$$

The quadratic term is squared Mahalanobis distance. The determinant $|\boldsymbol\Sigma|$ normalizes for the volume and orientation of the covariance ellipsoid. A single Gaussian is statistically efficient when the shape is plausible, but it cannot represent separated modes or strongly non-elliptical support. Gaussian mixtures add modes at the cost of more parameters and local optimization.

Maximum likelihood selects parameters that maximize

$$
\ell(\theta)=\sum_{i=1}^{n}\log p(\mathbf{x}_i\mid\theta).
$$

Evaluate density models on held-out observations. Training likelihood can always reward excess flexibility, including a Gaussian component collapsing around a point unless covariance is regularized.

#### **Kernel Density Estimation**

Kernel density estimation (KDE) places a smooth kernel around every observation and averages them:

$$
\widehat p_h(\mathbf{x})
=\frac{1}{nh^d}\sum_{i=1}^{n}
K\left(\frac{\mathbf{x}-\mathbf{x}_i}{h}\right).
$$

Here $K$ is a non-negative kernel integrating to one, $h>0$ is the bandwidth, and $h^d$ accounts for the rescaled $d$-dimensional volume. With a Gaussian kernel, KDE becomes a mixture of equal-covariance Gaussians centred on every observation.

<div class="diagram-scroll">

![How KDE bandwidth controls under-smoothing and over-smoothing.](assets/kde-bandwidth-bias-variance.svg){fig-alt="The same sample is shown with a small bandwidth producing noisy spikes, a moderate bandwidth preserving broad modes, and a large bandwidth merging structure."}

</div>

Bandwidth matters much more than the exact smooth kernel shape. A small $h$ creates low bias but high variance and can interpret sampling noise as modes. A large $h$ creates a stable but biased estimate that can erase meaningful modes. Cross-validated held-out log-likelihood is a defensible selector when density prediction is the objective.

KDE suffers acutely from the **curse of dimensionality**. The volume of a neighbourhood grows as $h^d$, so maintaining local resolution requires exponentially more observations as $d$ increases. Irrelevant features further dilute neighbourhoods. Dimension reduction can help only if it preserves the density features relevant to the task and its transformation is accounted for.

<details>
<summary><strong>Python: compare a Gaussian model with KDE on bimodal data</strong></summary>

```python
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.neighbors import KernelDensity

rng = np.random.default_rng(14)
X = np.concatenate([
    rng.normal(-2.2, 0.45, size=450),
    rng.normal(2.0, 0.85, size=550),
])[:, None]
X_train, X_test = train_test_split(X, test_size=0.35, random_state=14)

# Maximum-likelihood parameters for one univariate Gaussian.
mean = X_train.mean()
variance = X_train.var()
gaussian_log_density = -0.5 * (
    np.log(2 * np.pi * variance) + (X_test[:, 0] - mean) ** 2 / variance
)

kde = KernelDensity(kernel="gaussian", bandwidth=0.35).fit(X_train)
kde_log_density = kde.score_samples(X_test)

print("single Gaussian mean test log density:", round(gaussian_log_density.mean(), 3))
print("KDE mean test log density:", round(kde_log_density.mean(), 3))
```

</details>

<details>
<summary><strong>Python: select KDE bandwidth using cross-validated likelihood</strong></summary>

```python
import numpy as np
from sklearn.model_selection import GridSearchCV, KFold
from sklearn.neighbors import KernelDensity

rng = np.random.default_rng(30)
X = np.concatenate([
    rng.normal(-1.5, 0.35, size=220),
    rng.normal(1.3, 0.70, size=280),
])[:, None]

bandwidths = np.geomspace(0.05, 1.5, 18)
search = GridSearchCV(
    KernelDensity(kernel="gaussian"),
    param_grid={"bandwidth": bandwidths},
    cv=KFold(n_splits=5, shuffle=True, random_state=30),
)
search.fit(X)

print("selected bandwidth:", round(search.best_params_["bandwidth"], 3))
print("mean validation log-likelihood:", round(search.best_score_, 2))
```

</details>

The `GridSearchCV` score is total log-likelihood on a validation fold, so its magnitude depends on fold size. It is suitable for ranking bandwidths under the same split, not for comparison across arbitrary datasets.



### **Anomaly and Novelty Detection**

An **anomaly score** orders observations from more normal to more unusual under a specified model. Turning scores into binary alerts requires a threshold. This separation matters: a detector can rank rare events well while a poor threshold overwhelms analysts with false alerts.

Two deployment settings should be distinguished:

- **Outlier detection** fits a model to a training set that may already contain anomalies. Robustness to contamination is essential.
- **Novelty detection** fits only or mostly on representative normal data, then scores future observations. Training anomalies can distort the learned normal region.

An anomaly can be **global** (extreme relative to the whole sample), **local** (unusual only relative to nearby observations), **contextual** (unusual conditional on time, location, or subgroup), or **collective** (an unusual sequence even when individual values seem ordinary). No single point detector covers all four.

#### **Statistical Scores and Local Outlier Factor**

For approximately elliptical data, squared Mahalanobis distance provides a statistical score:

$$
s(\mathbf{x})
=(\mathbf{x}-\widehat{\boldsymbol\mu})^\top
\widehat{\boldsymbol\Sigma}^{-1}
(\mathbf{x}-\widehat{\boldsymbol\mu}).
$$

It corrects for feature scale and correlation, unlike raw Euclidean distance. Classical mean and covariance estimates are themselves sensitive to anomalies, so robust covariance estimators may be necessary. The Gaussian assumption also makes one global ellipse unsuitable for multimodal normal data.

Local Outlier Factor (LOF) asks whether a point has substantially lower local density than its neighbours. For neighbour $o$, define reachability distance

$$
\operatorname{reach\_dist}_k(p,o)
=\max\{d_k(o),d(p,o)\},
$$

where $d_k(o)$ is the distance from $o$ to its $k$-th neighbour. The local reachability density is the inverse average reachability distance from $p$ to its neighbour set $N_k(p)$:

$$
\operatorname{lrd}_k(p)
=\left(
\frac{1}{|N_k(p)|}
\sum_{o\in N_k(p)}\operatorname{reach\_dist}_k(p,o)
\right)^{-1}.
$$

LOF compares neighbour density with the point's density:

$$
\operatorname{LOF}_k(p)
=\frac{1}{|N_k(p)|}
\sum_{o\in N_k(p)}
\frac{\operatorname{lrd}_k(o)}{\operatorname{lrd}_k(p)}.
$$

Values near one indicate comparable local density; larger values suggest a local outlier. The neighbourhood size $k$ determines scale, and high-dimensional distance concentration can weaken the comparison.

#### **Isolation Forest and One-Class SVM**

Isolation Forest uses random axis-aligned splits. An unusual observation tends to be isolated after fewer splits than a point embedded in a dense region. Averaging path lengths across randomized trees yields a scalable anomaly score without explicitly estimating density. It works well in many tabular settings but can be less effective when useful anomalies require oblique, sequence, or context-dependent structure.

One-Class SVM learns a boundary around normal observations in a kernel feature space. With an RBF kernel, it can form a nonlinear normal region. The parameter $\nu$ controls a bound related to training errors and support vectors, while $\gamma$ controls kernel scale. Both the representation and feature scaling are critical; the method can be computationally expensive on large samples.

<div class="diagram-scroll">

![Global, local-density, isolation, and learned-boundary views of anomalies.](assets/anomaly-method-map.svg){fig-alt="Four panels compare Mahalanobis-style global distance, Local Outlier Factor, Isolation Forest random splits, and a One-Class SVM boundary."}

</div>

<details>
<summary><strong>Python: compare LOF, Isolation Forest, and One-Class SVM in novelty mode</strong></summary>

```python
import numpy as np
from sklearn.ensemble import IsolationForest
from sklearn.metrics import roc_auc_score
from sklearn.neighbors import LocalOutlierFactor
from sklearn.preprocessing import StandardScaler
from sklearn.svm import OneClassSVM

rng = np.random.default_rng(44)
# Fit only on representative normal observations.
normal_train = rng.normal(loc=[0, 0], scale=[1.0, 0.45], size=(700, 2))
normal_test = rng.normal(loc=[0, 0], scale=[1.0, 0.45], size=(300, 2))
anomaly_test = rng.uniform(low=[-6, -4], high=[6, 4], size=(70, 2))

X_test = np.vstack([normal_test, anomaly_test])
y_test = np.r_[np.zeros(len(normal_test)), np.ones(len(anomaly_test))]

scaler = StandardScaler().fit(normal_train)
X_train_scaled = scaler.transform(normal_train)
X_test_scaled = scaler.transform(X_test)

models = {
    "LOF": LocalOutlierFactor(n_neighbors=30, novelty=True).fit(X_train_scaled),
    "Isolation Forest": IsolationForest(
        n_estimators=300,
        random_state=44,
    ).fit(X_train_scaled),
    "One-Class SVM": OneClassSVM(kernel="rbf", gamma="scale", nu=0.05).fit(X_train_scaled),
}

for name, model in models.items():
    # sklearn scores are larger for more normal observations, so negate them.
    anomaly_score = -model.score_samples(X_test_scaled)
    print(name, "ROC AUC:", round(roc_auc_score(y_test, anomaly_score), 3))
```

</details>

ROC AUC measures ranking in this synthetic experiment and is threshold-independent. In deployment, alert precision, investigation capacity, missed-event cost, and delayed labels are usually more important than ROC AUC alone.

<details>
<summary><strong>Python: calibrate an alert threshold separately from model fitting</strong></summary>

```python
import numpy as np
from sklearn.ensemble import IsolationForest

rng = np.random.default_rng(52)
train_normal = rng.normal(size=(1000, 4))
calibration_normal = rng.normal(size=(500, 4))
future_batch = np.vstack([
    rng.normal(size=(195, 4)),
    rng.normal(loc=5.0, scale=0.5, size=(5, 4)),
])

detector = IsolationForest(n_estimators=300, random_state=52).fit(train_normal)
calibration_scores = -detector.score_samples(calibration_normal)
future_scores = -detector.score_samples(future_batch)

# This threshold targets a 1% alert rate on representative normal calibration data.
threshold = np.quantile(calibration_scores, 0.99)
alerts = future_scores >= threshold

print("calibrated threshold:", round(threshold, 3))
print("future alerts:", int(alerts.sum()), "of", len(alerts))
print("indices of largest scores:", np.argsort(future_scores)[-8:][::-1].tolist())
```

</details>

The quantile is an operational choice, not a guarantee that exactly one percent of future data are anomalous. Distribution shift can change the alert rate, so score distributions, alert volumes, and investigated outcomes should be monitored over time.



### **Evaluating Unsupervised Structure**

Unsupervised evaluation is difficult because the target that would define correctness is absent. A good evaluation therefore combines several independent lenses rather than treating one scalar index as ground truth.

#### **Internal, External, and Stability Criteria**

**Internal criteria** use the fitted representation, metric, and assignments. For observation $i$, let $a(i)$ be its mean distance to its own cluster and $b(i)$ the smallest mean distance to another cluster. Its silhouette is

$$
s(i)=\frac{b(i)-a(i)}{\max\{a(i),b(i)\}},
\qquad -1\leq s(i)\leq1.
$$

A high value means the observation is much closer to its assigned cluster than to the nearest alternative. The average silhouette rewards compact, separated clusters under the chosen distance. It tends to favor convex partitions and may penalize a scientifically valid curved or density-based structure.

The Calinski-Harabasz index compares between-cluster to within-cluster dispersion; larger is preferred. The Davies-Bouldin index averages each cluster's worst similarity to another cluster; smaller is preferred. All inherit assumptions from centroids and distances. They are diagnostics, not universal truth tests.

**External criteria** compare assignments with labels that were not used to fit the clusters. Adjusted Rand Index (ARI) measures pairwise agreement corrected for chance; Normalized Mutual Information (NMI) measures shared information. These can be valuable on simulated data or when independent expert categories exist. If labels are used repeatedly to select preprocessing, hyperparameters, and algorithms, the exercise has become supervised model selection and requires an untouched test set.

**Stability criteria** ask whether conclusions survive plausible perturbations: random initialization, bootstrap or subsampling, measurement noise, and modest hyperparameter changes. Because cluster names can permute, use label-invariant comparisons such as ARI or compare co-clustering matrices. A stable partition can still encode the wrong metric, while an unstable partition warns that substantive interpretation is fragile.

Density models should also be judged by held-out log-likelihood, calibration of simulated summaries, or task-specific predictive checks. Anomaly detectors with labels need precision-recall curves, recall at investigation capacity, time-aware validation, and post-alert review. Ultimately, **domain validity and downstream usefulness** remain separate from geometric neatness.

<div class="diagram-scroll">

![A four-lens framework for evaluating unsupervised results.](assets/unsupervised-evaluation.svg){fig-alt="Candidate structure is evaluated through internal geometry, external agreement, perturbation stability, and domain or downstream usefulness."}

</div>

<details>
<summary><strong>Python: show that an internal score can prefer the wrong geometry</strong></summary>

```python
from sklearn.cluster import KMeans, SpectralClustering
from sklearn.datasets import make_moons
from sklearn.metrics import adjusted_rand_score, silhouette_score
from sklearn.preprocessing import StandardScaler

X, synthetic_truth = make_moons(n_samples=700, noise=0.08, random_state=61)
X = StandardScaler().fit_transform(X)

models = {
    "K-Means": KMeans(n_clusters=2, n_init=20, random_state=61),
    "Spectral": SpectralClustering(
        n_clusters=2,
        affinity="nearest_neighbors",
        n_neighbors=14,
        n_init=20,
        random_state=61,
    ),
}

for name, model in models.items():
    labels = model.fit_predict(X)
    print(
        name,
        "| Euclidean silhouette:", round(silhouette_score(X, labels), 3),
        "| synthetic ARI:", round(adjusted_rand_score(synthetic_truth, labels), 3),
    )
```

</details>

The Euclidean silhouette evaluates compactness in the original coordinates, while the synthetic labels encode connectivity along each moon. A mismatch between the scores is expected because they ask different questions.

<details>
<summary><strong>Python: test assignment stability under small measurement perturbations</strong></summary>

```python
import numpy as np
from sklearn.cluster import KMeans
from sklearn.datasets import make_blobs
from sklearn.metrics import adjusted_rand_score
from sklearn.preprocessing import StandardScaler

X, _ = make_blobs(
    n_samples=600,
    centers=4,
    cluster_std=[0.7, 1.0, 0.8, 1.2],
    random_state=71,
)
X = StandardScaler().fit_transform(X)
base_labels = KMeans(n_clusters=4, n_init=30, random_state=71).fit_predict(X)

rng = np.random.default_rng(71)
stabilities = []
for repeat in range(30):
    perturbed = X + rng.normal(scale=0.03, size=X.shape)
    labels = KMeans(n_clusters=4, n_init=10, random_state=repeat).fit_predict(perturbed)
    stabilities.append(adjusted_rand_score(base_labels, labels))

print("mean perturbation stability:", round(float(np.mean(stabilities)), 3))
print("minimum perturbation stability:", round(float(np.min(stabilities)), 3))
```

</details>

Perturbation scale should reflect plausible measurement uncertainty, not an arbitrary amount chosen after seeing the answer. Report the stability distribution and where assignments change, not only its mean.



### **Choosing an Unsupervised Method**

Method selection should begin with the scientific or operational question. “Run clustering” is not yet a well-posed request. Specify what constitutes an observation, which variations should count as similar, whether every observation must belong to a group, and how the result will be used.

<div class="diagram-scroll">

![A decision map for selecting clustering, density, and anomaly methods.](assets/unsupervised-method-selection.svg){fig-alt="A decision flow starts from the desired output and routes centroid, connectivity, density, graph, probabilistic, and anomaly assumptions to suitable method families, followed by validation."}

</div>

| Situation | Strong starting point | Why | Diagnostic priority |
|---|---|---|---|
| Large numeric data with compact groups | K-Means or Mini-Batch K-Means | Fast and interpretable centroid baseline | Scaling, multiple starts, stability over $K$ |
| Need nested structure at moderate $n$ | Agglomerative clustering | Exposes several resolutions | Linkage sensitivity and dendrogram cuts |
| Irregular groups plus possible noise | DBSCAN | Encodes density connectivity directly | `eps`, `min_samples`, noise interpretation |
| Groups occur at varying densities | HDBSCAN | Selects persistent branches across scales | Minimum cluster size and stability |
| Similarity is naturally a graph | Spectral clustering | Uses connectivity rather than centroid geometry | Affinity construction and scalability |
| Need soft assignments or a generative density | Gaussian mixture | Probabilistic membership and covariance structure | Initialization, covariance type, held-out likelihood |
| Need a smooth low-dimensional density | KDE | Flexible nonparametric estimate | Bandwidth and held-out likelihood |
| Need unusual-observation ranking | LOF, Isolation Forest, or One-Class SVM | Different local, isolation, and boundary assumptions | Time-aware validation and threshold costs |

A practical workflow is:

1. **Define the unit and purpose.** Decide whether rows are people, sessions, documents, machines, or time windows, and what decision the output will support.
2. **Construct a defensible representation.** Handle units, skew, missingness, mixed types, and irrelevant features. Choose a metric whose changes have domain meaning.
3. **Inspect simple baselines.** Compare K-Means, a hierarchy, or a one-Gaussian density before introducing a flexible method.
4. **Vary assumptions deliberately.** Change scales, linkage, neighbourhood sizes, component counts, random seeds, and plausible preprocessing.
5. **Evaluate through several lenses.** Combine internal geometry, stability, held-out density, independent labels where legitimate, and domain review.
6. **Inspect records, not only plots.** Two-dimensional projections can distort high-dimensional neighbourhoods. Return to original features and representative examples.
7. **Document uncertainty and noise.** Avoid converting every algorithmic component into a confident semantic category.

<details>
<summary><strong>Python: create a compact, assumption-aware comparison report</strong></summary>

```python
import numpy as np
from sklearn.cluster import DBSCAN, KMeans
from sklearn.datasets import make_moons
from sklearn.metrics import silhouette_score
from sklearn.mixture import GaussianMixture
from sklearn.preprocessing import StandardScaler

X, _ = make_moons(n_samples=650, noise=0.08, random_state=90)
X = StandardScaler().fit_transform(X)

candidate_labels = {
    "K-Means": KMeans(n_clusters=2, n_init=30, random_state=90).fit_predict(X),
    "DBSCAN": DBSCAN(eps=0.20, min_samples=9).fit_predict(X),
    "GMM": GaussianMixture(n_components=2, n_init=10, random_state=90).fit_predict(X),
}

for name, labels in candidate_labels.items():
    assigned = labels != -1
    unique_assigned = np.unique(labels[assigned])
    # Silhouette requires at least two assigned groups and excludes DBSCAN noise here.
    if len(unique_assigned) >= 2 and assigned.sum() > len(unique_assigned):
        silhouette = silhouette_score(X[assigned], labels[assigned])
    else:
        silhouette = np.nan
    print(
        f"{name:8s}",
        "clusters=", len(unique_assigned),
        "noise_fraction=", round(float((~assigned).mean()), 3),
        "assigned_silhouette=", round(float(silhouette), 3),
    )
```

</details>

This report intentionally does not announce a winner. The silhouette uses Euclidean compactness and excludes DBSCAN noise, so it must be read together with the method's assumptions and the records left unassigned. A fair report states such choices explicitly.

The most important unsupervised habit is intellectual restraint. Algorithms will return clusters, scores, and density curves even when the sample contains weak, unstable, or irrelevant structure. The analyst's job is not to force a story from every output, but to establish which conclusions survive alternative representations, methods, samples, and substantive scrutiny.

Official implementations and worked comparisons are available in the [scikit-learn clustering guide](https://scikit-learn.org/stable/modules/clustering.html), [density-estimation guide](https://scikit-learn.org/stable/modules/density.html), and [novelty and outlier detection guide](https://scikit-learn.org/stable/modules/outlier_detection.html).
